### Testing knn accuracy of TF-IDF models

In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import datasets
import src
from src.knn_accuracy import knn_accuracy
from mteb.evaluation.evaluators.utils import get_vocab

2024-12-16 14:27:13.334003: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-16 14:27:13.358015: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-16 14:27:13.365465: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-16 14:27:13.397937: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-16 14:27:15.278641: W tensorflow/compiler/tf2

In [2]:
# load datasets
data_batched = {
"arxiv": datasets.load_dataset("mteb/arxiv-clustering-p2p", revision="a122ad7f3f0291bf49cc6f4d32aa80929df69d5d")["test"],
"biorxiv": datasets.load_dataset("mteb/biorxiv-clustering-p2p", revision="f5dbc242e11dd8e24def4c4268607a49e02946dc")["test"],
"medrxiv": datasets.load_dataset("mteb/medrxiv-clustering-p2p", revision="e7a26af6f3ae46b30dde8737f02c07b1505bcc73")["test"],
"reddit": datasets.load_dataset("mteb/reddit-clustering-p2p", revision="385e3cb46b4cfa89021f56c4380204149d0efe33")["test"],
"stackexchange": datasets.load_dataset("mteb/stackexchange-clustering-p2p", revision="815ca46b2622cec33ccafc3735d572c266efdb44")["test"]
}

In [3]:
data_full = {}
for name, data in data_batched.items():
    if name == "biorxiv":
        labels = [split["labels"] for split in data]
        sentences = [split["sentences"] for split in data]
    else:    
        labels = [x for split in data for x in split["labels"]]
        sentences = [x for split in data for x in split["sentences"]]
    data_full[name] = {"sentences": sentences, "labels": labels}

dataframe
colums: models
rows: datasets (full and batched)

Models to test:
- tfidf_log 
- tfidf_svd_log
- tfidf_svd_log_piecewise

#### logarithmic TF-IDF

In [ ]:
model = src.tfidf_log.Tfidf()
scores = {}

# get knn acc for full data 
for name, data in data_full.items():
    print(name)
    if name in ["arxiv", "reddit"]:
        # arxiv and reddit are too big to perform clustering on full tfidf representations
        continue            
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

# get knn acc for batched data
for name, data in data_batched.items():
    if name in ["biorxiv", "reddit"]: # biorxiv is the only one without batches
        continue
    # get full data to compute unified vocabulary
    vocab = get_vocab(data_full[name]["sentences"])
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores


with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)


#### logarithmic TF-IDF with svd reduction

In [5]:
def knn_acc_svd(model, dataset = data_full, dataset_batched = data_batched):

    scores = {}
    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)
        # compute unified vocabulary (only necessary because the svd models need a vocab input)
        vocab = get_vocab(data["sentences"])
        # compute svd components
        v = model.encode(sentences=data["sentences"], vocab = vocab)
        # compute svd reduced embeddings
        embeddings = model.encode(sentences=data["sentences"], vocab = vocab, V = v)
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        if name == "biorxiv": # biorxiv is the only one without batches
            continue
        # get full data to compute unified vocabulary
        vocab = get_vocab(dataset[name]["sentences"])
        # compute svd components
        v = model.encode(sentences=dataset[name]["sentences"], vocab = vocab)
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab, V = v)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
        scores[name + "_batchwise"] = batch_scores
    

    return scores


In [ ]:
model = src.tfidf_svd_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


model = src.tfidf_svd50_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


model = src.tfidf_svd200_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 


model = src.tfidf_svd300_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4)


model = src.tfidf_svd500_log.Tfidf()
scores = knn_acc_svd(model)
with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(732723, 331735)
V not provided, fit SVD to get V


In [10]:
def knn_acc(model, dataset = data_full, dataset_batched = data_batched):

    scores = {}

    # get knn acc for full data 
    for name, data in dataset.items():
        print(name)
        if name in ["arxiv", "reddit"]:
            # arxiv and reddit are too big to perform clustering on full tfidf representations
            continue            
        embeddings = model.encode(sentences=data["sentences"])
        scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

    # get knn acc for batched data
    for name, data in dataset_batched.items():
        print(name)
        if name in ["biorxiv"]: # biorxiv is the only one without batches
            continue
        # get full data to compute unified vocabulary
        vocab = get_vocab(data_full[name]["sentences"])
        # save scores for each batch
        batch_scores = []
        # iterate over batches
        for split in data:
            embeddings = model.encode(sentences=split["sentences"], vocab = vocab)
            batch_scores += [knn_accuracy(embeddings, split["labels"])]

        scores[name + "_batchwise"] = batch_scores
    
    return scores

#### TF-IDF with svd for each batch (but vocab for the entire set)

In [11]:
model = src.tfidf_svd_log_old.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)
dense matrix shape(53787, 100)
medrxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(37500, 73110)
dense matrix shape(37500, 100)
reddit
stackexchange
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(75000, 120133)
dense matrix shape(75000, 100)
arxiv
get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense mat

#### TF-IDF with svd with V and vocab for the each batch

In [12]:
model = src.tfidf_svd_log_old.Tfidf()
scores = {}

# get knn acc for full data 
for name, data in data_full.items():
    print(name)
    if name in ["arxiv"]:
        # arxiv and reddit are too big to perform clustering on full tfidf representations
        continue            
    embeddings = model.encode(sentences=data["sentences"])
    scores[name + "_full"] = knn_accuracy(embeddings, data["labels"])

# get knn acc for batched data
for name, data in data_batched.items():
    print(name)
    if name in ["biorxiv"]: # biorxiv is the only one without batches
        continue
    # save scores for each batch
    batch_scores = []
    # iterate over batches
    for split in data:
        embeddings = model.encode(sentences=split["sentences"])
        batch_scores += [knn_accuracy(embeddings, split["labels"])]
    
    scores[name + "_batchwise"] = batch_scores


with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}_novocab.json", "w") as fp:
    json.dump(scores , fp, indent = 4)

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)
dense matrix shape(53787, 100)
medrxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(37500, 73110)
dense matrix shape(37500, 100)
reddit
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(459399, 320065)
dense matrix shape(459399, 100)
stackexchange
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(75000, 120133)
dense matrix shape(75000, 100)
arxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(25000, 62013)
dense matrix shape(25000, 100)
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(25000, 62066)
dense matrix shape(25000, 100)
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(25000, 62001)
dense matrix shape(25000, 100)
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matr

#### TF-IDF with random projections

In [13]:
model = src.tfidf_rnd100_log.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)
dense matrix shape(53787, 100)
medrxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(37500, 73110)
dense matrix shape(37500, 100)
reddit
stackexchange
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(75000, 120133)
dense matrix shape(75000, 100)
arxiv
get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 100)
tfidf matrix shape(25000, 331735)
dense mat

In [14]:
model = src.tfidf_rnd768_log.Tfidf()
scores = knn_acc(model)

with open(f"text_embedding/MTEB/knn_results/{model.mteb_model_meta.name}_{model.mteb_model_meta.revision}.json", "w") as fp:
    json.dump(scores , fp, indent = 4) 

arxiv
biorxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(53787, 168265)
dense matrix shape(53787, 768)
medrxiv
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(37500, 73110)
dense matrix shape(37500, 768)
reddit
stackexchange
no vocab provided, computed by sklearns TfidfVectorizer call
tfidf matrix shape(75000, 120133)
dense matrix shape(75000, 768)
arxiv
get_vocab called!
input: <class 'list'> of length 732723
vocab of lenght 331735 starting with ['0000011d', '000026yorke', '00007h', '0001035v2', '00015t', '00019d', '0002bx', '0002d', '0002ev', '0002msun']
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 768)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 768)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 768)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 768)
tfidf matrix shape(25000, 331735)
dense matrix shape(25000, 768)
tfidf matrix shape(25000, 331735)
dense mat